# Guide 11 — Reading Number Hints (MNIST)

> **PYNQ Bootcamp guide.** This notebook takes one piece of the big competition program and explains it in small steps. Almost all of the code here is the *real* code that runs during a match — we've just split it up and added plain-English notes so it's easy to follow. (The one exception is the *Matching Strategy* guide, where the game plan is written as pseudocode for you to think through.)

## What is this notebook about?

You can "pay" for a hint that tells you where a card is — but the referee sends the
row and column as **pictures of handwritten numbers**, not as text! So the board uses
a second AI model called **MNIST** (a famous digit-reader) to read those number
pictures.

Big thing to notice: the chip can only hold **one** AI model at a time. So after
reading a number with MNIST, the code has to reload the YOLO "eye" before it can read
cards again. Forgetting that step is a classic bug.

All real code.


### How this guide fits in

**Depends on:** Guides 1 and 2 (it shares the same chip as the YOLO eye). **Used by:** Guide 12's hint handling.

*New here? Read **Guide 0 — How Everything Connects** first for the big picture.*


### Finding the MNIST model

Where the digit-reader model lives, plus a small math helper (`softmax` turns scores
into confidence percentages).


In [ ]:
MNIST_MODEL_PATHS = [
    NOTEBOOK_DIR / 'dpu_mnist_classifier.xmodel',
    Path('/home/root/jupyter_notebooks/PYNQ_Bootcamp/bootcamp_sessions/PYNQ 201 - MNIST/dpu_mnist_classifier.xmodel'),
    Path('/home/root/jupyter_notebooks/pynq-dpu/dpu_mnist_classifier.xmodel'),
]
MNIST_DIGITS_DIR_CANDIDATES = [
    NOTEBOOK_DIR / 'mnist_digits_0-9',
    Path('/home/root/jupyter_notebooks/PYNQ_Bootcamp/bootcamp_sessions/mnist_digits_0-9'),
]
mnist_runner = None
mnist_input_data = None
mnist_output_data = None
mnist_output_size = None
mnist_camera = None
mnist_camera_device = None
paid_hints = {}       # pos -> card name, e.g. {'B3': 'dog'}
paid_hint_state = {'active': False, 'name': '', 'row': None}


def mnist_model_path():
    for path in MNIST_MODEL_PATHS:
        if path.exists():
            return path
    raise FileNotFoundError('Could not find dpu_mnist_classifier.xmodel. Check the PYNQ 201 - MNIST notebook folder.')


def mnist_digits_dir():
    for path in MNIST_DIGITS_DIR_CANDIDATES:
        if path.exists():
            return path
    raise FileNotFoundError(
        'Could not find mnist_digits_0-9 folder. Copy it into bootcamp_sessions or this notebook folder.'
    )


def calculate_mnist_softmax(data):
    result = np.exp(data)
    return result / np.sum(result)

### Switching between MNIST and YOLO

`ensure_mnist_runner` loads the digit reader. `ensure_yolo_runner` reloads the card
"eye" afterward. You **must** switch back to YOLO after reading a digit — that's what
`ensure_yolo_runner` is for.


In [ ]:
def ensure_mnist_runner():
    global mnist_runner, mnist_input_data, mnist_output_data, mnist_output_size
    if mnist_runner is not None:
        return mnist_runner

    overlay.load_model(str(mnist_model_path()))
    mnist_runner = overlay.runner
    input_tensors = mnist_runner.get_input_tensors()
    output_tensors = mnist_runner.get_output_tensors()
    shape_in = tuple(input_tensors[0].dims)
    shape_out = tuple(output_tensors[0].dims)
    mnist_output_size = int(output_tensors[0].get_data_size() / shape_in[0])
    mnist_input_data = [np.empty(shape_in, dtype=np.float32, order='C')]
    mnist_output_data = [np.empty(shape_out, dtype=np.float32, order='C')]
    return mnist_runner


def ensure_yolo_runner():
    """Reloads the YOLO model after an MNIST capture swapped the DPU over --
    call this before any further board detection."""
    global dpu, inputTensors, outputTensors, shapeIn, shapeOut0, shapeOut1, shapeOut2
    global input_data, output_data, image_buffer, mnist_runner
    overlay.load_model(str(MODEL_PATH))
    dpu = overlay.runner
    inputTensors = dpu.get_input_tensors()
    outputTensors = dpu.get_output_tensors()
    shapeIn = tuple(inputTensors[0].dims)
    shapeOut0 = tuple(outputTensors[0].dims)
    shapeOut1 = tuple(outputTensors[1].dims)
    shapeOut2 = tuple(outputTensors[2].dims)
    input_data = [np.empty(shapeIn, dtype=np.float32, order='C')]
    output_data = [
        np.empty(shapeOut0, dtype=np.float32, order='C'),
        np.empty(shapeOut1, dtype=np.float32, order='C'),
        np.empty(shapeOut2, dtype=np.float32, order='C'),
    ]
    image_buffer = input_data[0]
    mnist_runner = None
    return dpu

### Reading a digit

These functions clean up the number picture (make it look like MNIST expects), run
the model, and give back the digit 0–9. `decode_digit_png_base64` reads the exact
picture the referee sends for a hint.


In [ ]:
def preprocess_mnist_frame(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    resized = cv2.resize(blurred, (28, 28), interpolation=cv2.INTER_AREA)

    # MNIST training data is white digit on black background. Invert bright paper backgrounds.
    if np.mean(resized) > 127:
        resized = 255 - resized

    normalized = np.asarray(resized / 255.0, dtype=np.float32)
    return np.expand_dims(normalized, axis=2)


def open_mnist_camera(device=None):
    global mnist_camera, mnist_camera_device
    if mnist_camera is not None and mnist_camera.isOpened():
        mnist_camera.release()

    if device is None:
        device = mnist_camera_device
    if device is None:
        candidates = [candidate for candidate in video_devices() if candidate != globals().get('camera_device')]
        device = candidates[0] if candidates else globals().get('camera_device')
    if device is None:
        raise RuntimeError('No MNIST camera device selected.')

    mnist_camera = open_camera(device)
    mnist_camera_device = device
    return mnist_camera


def read_mnist_camera_frame(device=None, read_attempts=8):
    if device is not None or mnist_camera is None or not mnist_camera.isOpened():
        open_mnist_camera(device)

    for _ in range(max(1, int(read_attempts))):
        ret, frame = mnist_camera.read()
        if ret and frame is not None:
            return frame
        mnist_camera.grab()
        ret, frame = mnist_camera.retrieve()
        if ret and frame is not None:
            return frame
        time.sleep(0.05)

    open_mnist_camera(device)
    ret, frame = mnist_camera.read()
    if ret and frame is not None:
        return frame
    raise RuntimeError('Could not read a frame from the MNIST camera.')


def run_mnist_model_from_processed(digit_image):
    runner = ensure_mnist_runner()
    mnist_input_data[0][0, ...] = digit_image.reshape(mnist_input_data[0].shape[1:])
    job_id = runner.execute_async(mnist_input_data, mnist_output_data)
    runner.wait(job_id)
    logits = mnist_output_data[0].reshape(1, mnist_output_size)[0]
    probabilities = calculate_mnist_softmax(logits)
    prediction = int(probabilities.argmax())
    return prediction, probabilities


def _show_mnist_prediction(frame, digit_image, prediction, confidence):
    annotated = draw_label(frame.copy(), f'MNIST prediction: {prediction} ({confidence:.2f})', frame.shape[1] // 2, 30, (0, 255, 0))
    _set_image_widget(mnist_image_widget, annotated)


def predict_mnist_digit_from_frame(frame, show=True):
    digit_image = preprocess_mnist_frame(frame)
    prediction, probabilities = run_mnist_model_from_processed(digit_image)
    confidence = float(probabilities[prediction])

    if show:
        _show_mnist_prediction(frame, digit_image, prediction, confidence)

    # Restore YOLO so normal board detection still works after this capture.
    ensure_yolo_runner()
    print(f'MNIST prediction: {prediction} (confidence={confidence:.3f})')
    return prediction


def capture_mnist_digit(device=None, show=True):
    frame = read_mnist_camera_frame(device=device)
    return predict_mnist_digit_from_frame(frame, show=show)


def decode_digit_png_base64(png_base64, show=True):
    """Decodes a base64 PNG digit image (as delivered by hint_response)
    into a predicted digit 0-9 via the on-board MNIST classifier."""
    png_bytes = base64.b64decode(png_base64)
    frame = cv2.imdecode(np.frombuffer(png_bytes, dtype=np.uint8), cv2.IMREAD_COLOR)
    if frame is None:
        raise ValueError('Could not decode the hint digit image.')
    return predict_mnist_digit_from_frame(frame, show=show)


def predict_mnist_digit_from_saved_image(digit_choice, show=True):
    """Debug-only: classify a pre-saved sample digit image from
    mnist_digits_0-9/ instead of the live MNIST camera -- lets you test the
    Paid Hint capture flow without a working MNIST camera or writing digits
    by hand."""
    image_path = mnist_digits_dir() / f'digit_{int(digit_choice)}.png'
    frame = cv2.imread(str(image_path))
    if frame is None:
        raise FileNotFoundError(f'Could not read saved digit image: {image_path}')
    return predict_mnist_digit_from_frame(frame, show=show)

### Saving the hint

Once we know the row and column, `inject_paid_hint` saves *"this card is at B3"* so
the game plan can use it.


In [ ]:
def inject_paid_hint(card_name, row, col):
    card_name = str(card_name).strip()
    row = int(row)
    col = int(col)
    if not card_name:
        raise ValueError('Enter a card name before using Paid Hint.')
    if not (0 <= row < GRID_ROWS and 0 <= col < GRID_COLS):
        raise ValueError(f'Hint digits row={row} col={col} are outside the {GRID_ROWS}x{GRID_COLS} grid.')

    pos = pos_name(row, col)
    paid_hints[pos] = card_name
    print(f'Paid hint stored: {card_name} at {pos}')
    return {'pos': pos, 'description': card_name}

### The buttons for hints

This last block builds the little control panel for capturing hints: pick a camera,
type the card name, then capture the row digit and the column digit. It plugs into
the main control panel from Guide 10. All real code — skim it.


In [ ]:
_mnist_camera_options = video_devices() or ['(no camera found)']
mnist_camera_dropdown = widgets.Dropdown(
    options=_mnist_camera_options,
    value=next((d for d in _mnist_camera_options if d != camera_device), _mnist_camera_options[0]),
    description='MNIST cam:',
)
mnist_digit_source_toggle = widgets.ToggleButtons(
    options=[('Live Camera', 'camera'), ('Saved Image', 'saved')],
    value='camera',
    description='Digit source:',
)
saved_digit_dropdown = widgets.Dropdown(options=list(range(10)), value=0, description='Saved digit:')

mnist_image_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='416px'))
mnist_output = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '8px'})

paid_hint_name_text = widgets.Text(value='', placeholder='card name, e.g. car', description='Hint name:')
paid_hint_button = widgets.Button(description='Paid Hint', button_style='warning')
paid_hint_capture_button = widgets.Button(description='Capture Row', button_style='warning', disabled=True)
paid_hint_capture_button.layout.display = 'none'
mnist_debug_button = widgets.Button(description='Debug MNIST One-Shot', button_style='primary')
paid_hints_show_button = widgets.Button(description='Show Paid Hints', button_style='info')
paid_hints_clear_button = widgets.Button(description='Clear Paid Hints', button_style='danger')


def _run_mnist_action(label, action):
    with mnist_output:
        clear_output(wait=True)
        print(label)
        try:
            action()
        except Exception:
            import traceback
            traceback.print_exc()


def _reset_paid_hint_capture_button():
    paid_hint_state['active'] = False
    paid_hint_state['name'] = ''
    paid_hint_state['row'] = None
    paid_hint_capture_button.description = 'Capture Row'
    paid_hint_capture_button.disabled = True
    paid_hint_capture_button.layout.display = 'none'


def on_paid_hint_clicked(_button):
    def action():
        card_name = paid_hint_name_text.value.strip()
        if not card_name:
            raise ValueError('Enter a card name before clicking Paid Hint.')
        paid_hint_state['active'] = True
        paid_hint_state['name'] = card_name
        paid_hint_state['row'] = None
        paid_hint_capture_button.description = 'Capture Row'
        paid_hint_capture_button.disabled = False
        paid_hint_capture_button.layout.display = ''
        print(f"Paid hint started for '{card_name}'. Capture the row digit, then the column digit.")
    _run_mnist_action('Paid Hint', action)


def on_paid_hint_capture_clicked(_button):
    def action():
        if not paid_hint_state['active']:
            raise RuntimeError('Click Paid Hint first.')

        if mnist_digit_source_toggle.value == 'saved':
            digit = predict_mnist_digit_from_saved_image(saved_digit_dropdown.value, show=True)
        else:
            digit = capture_mnist_digit(device=mnist_camera_dropdown.value, show=True)

        if paid_hint_state['row'] is None:
            paid_hint_state['row'] = digit
            paid_hint_capture_button.description = 'Capture Column'
            print(f'Captured row: {digit}. Now capture the column digit.')
            return

        row = paid_hint_state['row']
        col = digit
        result = inject_paid_hint(paid_hint_state['name'], row, col)
        _reset_paid_hint_capture_button()
        print(result)
    _run_mnist_action('Capture Digit', action)


def on_mnist_debug_clicked(_button):
    def action():
        capture_mnist_digit(device=mnist_camera_dropdown.value, show=True)
    _run_mnist_action('Debug MNIST One-Shot', action)


def on_paid_hints_show_clicked(_button):
    def action():
        if not paid_hints:
            print('No paid hints stored yet.')
        for pos, description in sorted(paid_hints.items()):
            print(f'  {pos}: {description}')
    _run_mnist_action('Paid Hints', action)


def on_paid_hints_clear_clicked(_button):
    def action():
        paid_hints.clear()
        print('Paid hints cleared.')
    _run_mnist_action('Clear Paid Hints', action)


paid_hint_button.on_click(on_paid_hint_clicked)
paid_hint_capture_button.on_click(on_paid_hint_capture_clicked)
mnist_debug_button.on_click(on_mnist_debug_clicked)
paid_hints_show_button.on_click(on_paid_hints_show_clicked)
paid_hints_clear_button.on_click(on_paid_hints_clear_clicked)

mnist_controls = widgets.VBox([
    widgets.HTML('<h3>MNIST Paid-Hint Debug</h3>'),
    widgets.HBox([mnist_camera_dropdown, mnist_digit_source_toggle, saved_digit_dropdown]),
    widgets.HBox([mnist_debug_button]),
    mnist_image_widget,
    widgets.HTML('<small>Type a card name, click Paid Hint, then capture the row digit and the column digit. Switch to Saved Image to test with mnist_digits_0-9/ instead of a live camera.</small>'),
    widgets.HBox([paid_hint_name_text, paid_hint_button, paid_hint_capture_button]),
    widgets.HBox([paid_hints_show_button, paid_hints_clear_button]),
    mnist_output,
])

# Keep every interactive control in the one Section 10 dashboard.
extended_tools_panel.children = (mnist_controls,)

### Check yourself

1. Why do we have to reload YOLO right after reading a number with MNIST?
2. The referee sends the hint as a *picture* of a number. Why can't we just read it
   as text?
